# A2LLM - train ALL models on MULTILINGUAL billion-token corpus (T4)

**Corpus:** en (wikitext-103 + TinyStories) + hi + ur + bn (Common Crawl) ~ 2.2 GB = **~2.2 B tokens**

1. Runtime → Change runtime type → **T4 GPU** (free)
2. **Runtime → Run all** — sab kuch sequential chalega (total ~3 h, session 12 h limit ke andar (PC band kar sakte ho - Colab cloud hai))
3. Ya har model ka cell alag-alag chalao (har cell ke baad download kar sakte ho)
4. Last cell sab checkpoints zip karke download karta hai

**Steps formula:** 1 epoch = 2.2e9 / (batch_size × ctx)  |  steps yahan ~0.75-0.9 epoch ke liye set hain

In [ ]:
!git clone https://github.com/ayushrajdev9-cmyk/a2llm
%cd /content/a2llm/a2llm-esp32

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu124
!pip install -q numpy pytest

In [ ]:
# DOWNLOAD + BUILD multilingual corpus (~2.2 GB, ~5-8 min)
!python scripts/prepare_data_large.py

## 1) NANO - ESP32 tier (34K params) ~ 36 min

In [ ]:
# ~0.9 epoch: 2.2e9 / (128*32) = 537K full -> 250K steps (ctx 32, batch 128)
!python scripts/train.py --preset nano --data data/pretrain_multilingual.txt --steps 250000 --batch-size 128 --out checkpoints/nano
!python scripts/generate.py checkpoints/nano/best.pt --prompt "ROMEO:" --max-new-tokens 80

## 2) MICRO - laptop tier (120K params) ~ 2 h

In [ ]:
# ~0.75 epoch: 2.2e9 / (128*64) = 268K full -> 200K steps (ctx 64, batch 128)
!python scripts/train.py --preset micro --data data/pretrain_multilingual.txt --steps 200000 --batch-size 128 --out checkpoints/micro
!python scripts/generate.py checkpoints/micro/best.pt --prompt "ROMEO:" --max-new-tokens 80

## 3) MINI - default PC model (1.06M params) ~ 1 h

In [ ]:
# ~0.75 epoch: 2.2e9 / (128*128) = 134K full -> 100K steps (ctx 128, batch 128)
!python scripts/train.py --preset mini --data data/pretrain_multilingual.txt --steps 100000 --batch-size 128 --out checkpoints/mini
!python scripts/generate.py checkpoints/mini/best.pt --prompt "ROMEO:" --max-new-tokens 80

## 4) SMALL - better PC / entry GPU (2.76M params) ~ 2.5 h

In [ ]:
# ~0.75 epoch: 2.2e9 / (128*256) = 67K full -> 50K steps (ctx 256, batch 128)
!python scripts/train.py --preset small --data data/pretrain_multilingual.txt --steps 50000 --batch-size 128 --out checkpoints/small
!python scripts/generate.py checkpoints/small/best.pt --prompt "ROMEO:" --max-new-tokens 80

## 5) Download all checkpoints

In [ ]:
!zip -r a2llm-all-best.zip checkpoints/nano/best.pt checkpoints/micro/best.pt checkpoints/mini/best.pt checkpoints/small/best.pt
from google.colab import files
files.download("a2llm-all-best.zip")